# 第2回 宿題（発展）：用語集のSFTと検索の比較

指定の `scripts/make_sft_dataset.py` で生成した58語・116件のQ&Aを使います。元スクリプトの `messages` 形式（文章回答）をそのまま学習し、JSON形式遵守率は測りません。モデルには評価時に用語集本文を渡しません。

**手順**：このノートブックと同梱ZIPをダウンロードし、Colabにノートブックをアップロード。新しいT4 GPUセッションで0〜7を実行してください。セクション1でZIPをアップロードします。

学習はまず3エポックに設定しています。元の14件よりデータ量が多いため、所要時間は実測してください。学習前後と用語名・別名検索を同じ18問で比較します。既知語を言い換えた評価であり、未学習の既知語への汎化を測る分割ではありません。用語集にない実在語は一般知識として知っていても、今回の評価では確認を求めることを正解とします。未知語の確認要求は学習データに含まれないため、SFT後も保証されません。

## 0. GPUとライブラリの準備

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

In [ ]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. データのアップロードと設定
`glossary_advanced_colab.zip` を選んでください。入力は `/content/glossary_input`、結果は `/content/glossary_lora_lab` に保存します。

In [ ]:
from pathlib import Path
import json, time, zipfile, io, hashlib, unicodedata
from google.colab import files
uploaded = files.upload()
zip_names = [name for name in uploaded if name.endswith('.zip')]
assert len(zip_names) == 1, '配布したZIPを1つ選んでください'
INPUT = Path('/content/glossary_input'); INPUT.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(uploaded[zip_names[0]])) as z:
    # 必要な3ファイルだけを固定パスへ書き込む
    for name in ['glossary.json', 'glossary_sft.jsonl', 'glossary_eval.json']:
        (INPUT / name).write_bytes(z.read(name))
WORK = Path('/content/glossary_lora_lab'); WORK.mkdir(exist_ok=True)
CONFIG = dict(base_model='Qwen/Qwen3.5-0.8B', revision='2fc06364715b967f1860aea9cf38778875588b17',
              seed=42, num_train_epochs=3, warmup_steps=6, lr=2e-4, lora_r=16,
              lora_alpha=32, max_seq_len=512, max_new_tokens=256, gen={'do_sample': False})
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

## 2. 学習データと評価基準
116件をそのまま学習します。評価は用語名6問・別名と言い換え6問・用語名なし3問・用語集外3問です。評価質問は学習質問と完全一致しませんが、既知語の定義は学習済みです。

既知語は「必須概念の語句グループをすべて含むか」で自動採点します。これは補助指標で、矛盾や誤った補足までは検出しません。未知語は確認・不明を示す表現の有無を測ります。説明を断定しながら末尾で確認を求める回答も通り得るため、原文を必ず読み、正しい定義・誤った補足・適切な確認要求を手動で点検してください。

In [ ]:
records = [json.loads(line) for line in (INPUT/'glossary_sft.jsonl').read_text().splitlines() if line.strip()]
glossary = json.loads((INPUT/'glossary.json').read_text())['terms']
cases = json.loads((INPUT/'glossary_eval.json').read_text())
training_questions = {r['messages'][0]['content'] for r in records}
assert len(records) == 116 and len(glossary) == 58 and len(cases) == 18
assert all(c['question'] not in training_questions for c in cases)
assert len({c['question'] for c in cases}) == len(cases)
print('学習件数:', len(records), '| 用語:', len(glossary), '| 評価:', len(cases))

## 3. モデルの読み込みと学習前評価

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)


# 学習データと同じuserメッセージのみ。用語集本文や正解は渡さない。
def build_messages(example):
    return [{"role": "user", "content": example["question"]}]
def generate(model, example):
    # enable_thinking=False: 思考ブロックを空にして回答だけを生成させる（モデルを替えても条件が揃うように明示）
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
def norm(s):
    return ''.join(unicodedata.normalize('NFKC', s).casefold().split())
def score(raw, case):
    text = norm(raw)
    if not case['known']:
        return any(norm(w) in text for w in ['確認が必要', '確認できません', 'わかりません', '分かりません', '不明', '記載がありません'])
    return all(any(norm(w) in text for w in group) for group in case['keyword_groups'])
def evaluate_answerer(answerer, label):
    start = time.time(); rows = []
    for case in cases:
        raw = answerer(case)
        rows.append({**case, 'raw': raw, 'auto_pass': score(raw, case)})
    summary = {'label': label, 'n': len(rows),
               'auto_pass_rate': sum(r['auto_pass'] for r in rows)/len(rows),
               'seconds': round(time.time()-start, 1), 'by_category': {}}
    for category in dict.fromkeys(r['category'] for r in rows):
        group = [r for r in rows if r['category']==category]
        summary['by_category'][category] = {'n': len(group), 'auto_pass_rate': sum(r['auto_pass'] for r in group)/len(group)}
    (WORK/f'eval_{label}.json').write_text(json.dumps({'summary': summary, 'rows': rows}, ensure_ascii=False, indent=2))
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    display(pd.DataFrame(rows)[['category', 'question', 'reference', 'raw', 'auto_pass']])
    return summary, rows
before_summary, before_rows = evaluate_answerer(lambda c: generate(model, c), 'before')

## 4. LoRA学習
基盤モデルを固定し、元の演習と同じ対象層にLoRAを追加します。116件・3エポックです。ここでは検証セットによる設定選択は行いません。評価18問を見て設定を何度も選び直した場合、それらは開発用評価となるため、最終評価には別の質問を用意してください。

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
train_ds = Dataset.from_list(records)

# 切り詰めで正解末尾が欠けないよう、学習前に長さを検証する
lengths = [len(tok.apply_chat_template(r["messages"], tokenize=True, enable_thinking=False)) for r in records]
print("学習データ最大トークン数:", max(lengths))
assert max(lengths) <= CONFIG["max_seq_len"], "max_seq_lenを増やし、セッションを再起動して実行してください"

model.config.use_cache = False
# Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # 通常は変えない。True にすると計算だけ 16bit の混合精度になる（CUDA のみ。重みは 32bit のまま）。速くなる一方、損失が nan になる場合がある
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # LoRA で学習するパラメータの割合
t0 = time.time()
trainer.train()  # 途中で止まった場合の再開は 4b を参照
training_seconds = time.time()-t0
print(f"学習時間: {training_seconds:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # 学習モードのままだと dropout が効き、評価の出力が揺れる（代入はモデル全体の表示を抑えるため）

## 5. 学習後評価
学習前と同じ質問・生成条件で比較し、原文を確認します。

In [ ]:
after_summary, after_rows = evaluate_answerer(lambda c: generate(tuned, c), 'after')
regressed = [b['question'] for b,a in zip(before_rows,after_rows) if b['auto_pass'] and not a['auto_pass']]
print('自動採点で退行した質問:', regressed)
display(pd.DataFrame([{'質問': b['question'], '学習前': b['raw'], '学習後': a['raw']} for b,a in zip(before_rows,after_rows)]))

## 6. 用語名・別名による検索との比較
検索はモデルを呼ばず、質問に含まれる用語名・別名を探し、一致が最も長い候補の定義を返します。同じ長さで複数候補があれば確認を求めます。大文字小文字・空白・文字幅を正規化します。意味検索やRAG全体の評価ではありません。

定義を直接返すので一致したときの出典は明確ですが、用語名のない説明には対応できない場合があります。短い別名の偶然の一致や複数語の比較質問にも弱いため、検索結果のterm_idと実際の質問を照合してください。モデル側は定義を入力に持たず、検索側は用語集を参照できるという条件の違いも考察します。

In [ ]:
def retrieve(case):
    query = norm(case['question']); hits = []
    for term in glossary:
        matching = [norm(name) for name in [term['term'], *term.get('aliases', [])] if norm(name) in query]
        if matching: hits.append((max(map(len, matching)), term))
    if not hits: return None
    longest = max(length for length, _ in hits)
    winners = [term for length, term in hits if length==longest]
    return winners[0] if len(winners)==1 else None
retrieval_matches = [{'question': c['question'], 'retrieved_term_id': (retrieve(c) or {}).get('id')} for c in cases]
def search_answer(case):
    hit = retrieve(case)
    return hit['definition'] if hit else 'この用語集では確認できません。確認が必要です。'
search_summary, search_rows = evaluate_answerer(search_answer, 'search')
display(pd.DataFrame(retrieval_matches))
display(pd.DataFrame([before_summary, after_summary, search_summary])[['label','n','auto_pass_rate','seconds']])

## 7. 保存と提出
結果ZIPには設定、学習時間、最終記録Loss、入力データのハッシュ、評価基準、各方式の原文を含めます。`manual_review.csv` は空欄の手動点検表です。既知語の正しい言い換えを自動採点が落としていないか、未知語で断定していないかを読み、`manual_correct` と `notes` を記入してください。自動採点値を厳密な内容正解率として報告しないでください。

学習のメリット・デメリットを、言い換えへの対応、事実の正確さ、未知語への対応、更新時の再学習、学習・推論時間、検索の出典の明確さから考察します。結果を保存したらランタイムを接続解除して削除してください。

In [ ]:
loss_logs = [entry for entry in trainer.state.log_history if 'loss' in entry]
result = {'topic': '講座用語集のSFTと検索の比較', 'config': CONFIG,
          'versions': {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
          'method': 'LoRA（量子化なし・文章回答）', 'executed_training_myself': True,
          'training': {'n': len(records), 'seconds': round(training_seconds,1), 'steps': trainer.state.global_step,
                       'last_logged_loss': loss_logs[-1]['loss'] if loss_logs else None},
          'before': before_summary, 'after': after_summary, 'search': search_summary,
          'regressed': regressed, 'retrieval_matches': retrieval_matches,
          'input_sha256': {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in INPUT.iterdir() if p.is_file()},
          'metric_note': '自動採点は必須語句または確認表現の一致。意味の正確さは手動点検が必要。'}
(WORK/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2))
(WORK/'glossary_eval.json').write_text(json.dumps(cases,ensure_ascii=False,indent=2))
review = []
for label, rows in [('before', before_rows), ('after', after_rows), ('search', search_rows)]:
    for row in rows:
        review.append({'method': label, 'category': row['category'], 'question': row['question'],
                       'reference': row['reference'], 'raw': row['raw'], 'auto_pass': row['auto_pass'],
                       'manual_correct': '', 'notes': ''})
pd.DataFrame(review).to_csv(WORK/'manual_review.csv',index=False,encoding='utf-8-sig')
archive = WORK/'glossary_results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for name in ['result.json','eval_before.json','eval_after.json','eval_search.json','glossary_eval.json','manual_review.csv']:
        z.write(WORK/name,name)
files.download(str(archive))